In [1]:
# Algebraic treatment of a Symmetric Plate-Tensegrity-Structure with the 332 Rotational Spherical Symmetry of a Tetrahedron. 
# GT, 2026-04-20

# 1. Description of the Structure
# The structure consists of 4 x 3 ring cables and 6 edge-cables as well as 4 equilateral triangular plates. 
# Nodes are treated as points in 3D euclidean space, cables as lines between points and plates as 2D polygons. 
# At each node exactly one vertex of a plate and three cables meet. Two of the cables at each point are ring cables, one is an edge-cable. 
# By definition all twelve nodes or points are equal up to symmetry. 

# 2. What we want to derive 
# We want derive a general formula for the coordinates of one node in an euclidean coordinate system representing the a presstressable configuration of the tensegrity structure
# The free parameters of the formula then can be constrained further by choosing a scale or edge-lengths of the structure. 
# By introducing further constraints we want to derive specific solutions for the coordinates of the nodes as well as formulas representing trajectories of transformations that occur if cables are shortended or lengthened. 
# We visualize the structure and its transformations in 3D space parametrically as a function of both rotation angles of the plates and the proportions of ring- end edge-cables.
# For any given solution we visualize the force equilibrium at one node with a closed polygon of forces in 3d space.

# 3. Geometric Setup and Orientation of the Fundamental Region 
# For practial reasons we fix the fundamental region of the symmetry group r332 with its center at the origin.
# Moreover we align one of the axes of symmetry (a1) with the z-axis. 
# And the second axis of the fundamental region (a2) is in the xz-plane.
# Since dealing algebraically with the rotations in 3D is cumbersome in usual matrix formulation, we use geometric algebra to derive algebraic constraints from symmetry.
# In Geometric Algebra P(3,0,1) we can express all rotations as rotors in a very compact form.   

# 4. Programming Setup
# We use the Kingdon package for Python to perform algebraic operations in Geometric Algebra P(3,0,1).
# Sympy is used to perform general algebraic manipulations.
# Math in Python is used to perform numerical calculations for the visualization of specific solutions. 
# The Ipykernel and Ipywidgets and the Ipython.display packages are used for visualization. 



In [2]:
# Setting up the programming environment by importing the necessary packages.

from kingdon import Algebra
import sympy as sym
from math import cos, sin, pi, e, sqrt, acos, atan
import ipywidgets
from IPython.display import display
from ipywidgets import FloatSlider
import numpy as np
from scipy.optimize import root_scalar

In [3]:
# Setting up the Projective Algebra PGA (3,0,1)
alg = Algebra(3, 0, 1)
locals().update(alg.blades)

# For representation of points, lines and planes in PGA (3,0,1) please refer to the Geometric Algebra Cheat Sheet: https://bivector.net/3DPGA.pdf
# For the Kingdon-Syntax please refer to the documentation: https://kingdon.readthedocs.io/en/latest/

# Points represent nodes 
point = lambda x, y, z: (e0 + x*e1 + y*e2 + z*e3).dual()
# Lines represent cables with zero diameter
line = lambda x1, y1, z1:(e0 + x1*e12 + y1*e13 + z1*e23)
# Planes represent plates with zero thickness
plane = lambda a, b, c, d: alg.vector(e1=a, e2=b, e3=c, e0=d)


In [4]:
# We need some symbols for multivectors that represent axes and rotations
# a1, a2 and a3 represent the three axes of symmetry of the fundamental region all meeting at the origin.
# ra1, ra2 and ra3 represent the angles of rotation around the three axes.

a1 = sym.symbols('a1', real=True)
a2 = sym.symbols('a2', real=True)
a3 = sym.symbols('a3', real=True)
ra1 = sym.symbols('ra1', real=True)
ra2 = sym.symbols('ra2', real=True)
ra3 = sym.symbols('ra3', real=True)

# Mapping of axes to blades: e12: z-axis, e13: y-axis, e23: x-axis
# a1 represents the z-axis and is encoded in the blade e12 of PGA (3,0,1)

a1 = e12


In [5]:
# Now we derive the other two axes of symmetry of the fundamental region and the rotors representing the rotations around these axes.
# First rotor of 120° degrees around a1, ie. the z-axis of the fundamental region of the symmetry group r332 with 3-fold symmetry.
# Please note the rotors in PGA (3,0,1) need to defined by the half of the angle of rotation.

angle_1 = 4*sym.pi/3
ra1 = sym.cos(angle_1/2) +sym.sin(angle_1/2)*a1

# Second axis of rotation of the fundamental region with angle (acos -1/3)/2 = -54°:
# This angle is the angle between the z-axis and the second axis of rotation of the fundamental region.

angle_2 = sym.acos(sym.Rational(-1,3))/2
r2 = sym.cos(angle_2/2) + sym.sin(angle_2/2) *e13
a2 =  r2 >> a1

# Second rotor with two-fold symmetry around the second axis of rotation of the fundamental region.
ra2 = sym.cos(sym.pi/2)+ sym.sin(sym.pi/2)*a2

# Third axis of rotation of the fundamental region with 2-fold symmetry : we apply a double rotation for the construction
# rotation around y-axis with acos(1/3) = 70.53°
angle_2 = sym.acos(sym.Rational(-1,3))
r3 = sym.cos(-angle_2/2) + sym.sin(-angle_2/2) *e13
r3
a31 = r3 >> a1    

# rotation around z-axis with 60°
angle_3 = -sym.pi/3
angle_3
r4 = sym.cos(angle_3/2) + sym.sin(angle_3/2) * e12
a3 = r4 >> a31

# Third rotor of 120° around the third axis of rotation of the fundamental region
ra3 = sym.cos(-sym.pi/3)+ sym.sin(-sym.pi/3)*a3

# Origin
p0 = point (0,0,0)

# Symbols for x, y, z values of the point we are looking for. 
x = sym.symbols('x', real=True, positive= True)
y = sym.symbols('y', real=True, positive= True) 
z = sym.symbols('z', real=True, positive= True)

# We naming the points as P.. The first index refers to a ring the second index to a point on the ring. 
# "p11" is the first point on the first triangular cable-ring. 
# This will help us to keep track of the points at cable-rings later on. 
 
p11 = point (x,y,z)

In [6]:
# Having defined the first point as a variable of its three coordinates x,y,z we now can construct all other points using the three rotors of our symmetry group. 
# For this action of the symmetry group on a point we define a subroutine that takes a point as an argument and applies the three rotors to it to derive the coordinates of all other 11 points.
# Additionally we derive the coordinates of the centers of the first plate and the center of the first ring of three cables as well as we need those later.

def points_tensegrity_tetrahedron (node):

    p11s = node    

    # Doing rotations in Geometric Algebra is really simple.
    # Our naming convention now is mapping each point to a specific cable-ring

    # Points of cable-ring 1. 
    p21s = ra3 >> p11s
    p31s = ra3 >> p21s

    # Points of cable-ring 2
    p12s = ra1 >> p11s
    p22s = ra1 >> p21s
    p32s = ra1 >> p31s

    # Points of cable-ring 3
    p13s = ra1 >> p12s  
    p23s = ra1 >> p22s  
    p33s = ra1 >> p32s  

    # Points of cable-ring 4
    p34s = ra2 >> p12s
    p14s = ra1 >> p34s
    p24s = ra1 >> p14s

    # from the points or nodes we can easily derive cables and plates and midpoints as multivectors handily with GA by simple addition:
    # for algebraic reasons we do not normalize the points yet, but leave that for later.
    # please refer for the naming convention to the labled images that show up further down. 
    
    # Center of the frist ring of three cables 
    r1ms = p11s + p21s + p31s
        
    # Center of plate 1
    q1ms = p11s + p12s + p13s

    return p12s, p13s, p21s, p22s, p23s, p31s, p32s, p33s, p14s, p24s, p34s, r1ms, q1ms

# Now we can call the subroutine to get the coordinates of all points as algebraic expressions in x,y and z.

p12, p13, p21, p22, p23, p31, p32, p33, p14, p24, p34, r1m, q1m = points_tensegrity_tetrahedron (p11)





In [7]:
# Now we have algebraic expressions for twelve different points in 3D in the language of Geometric Algebra, but have only 3 variables x,y and z to solve for meaning our system has 3 degrees of freedom. 
# This algebraic setup of defining looks unfamliliar at first, but is actually a very good starting point for our further analysis.

display("p11 = " + str(p11))
display("p12 = " + str(p12))
display("p13 = " + str(p13))
display("p14 = " + str(p14))
display("p21 = " + str(p21))
display("p22 = " + str(p22))
display("p23 = " + str(p23))
display("p24 = " + str(p24))
display("p31 = " + str(p31))
display("p32 = " + str(p32))
display("p33 = " + str(p33))
display("p34 = " + str(p34))



'p11 = (-z) 𝐞₀₁₂ + y 𝐞₀₁₃ + (-x) 𝐞₀₂₃ + 1 𝐞₁₂₃'

'p12 = (-z) 𝐞₀₁₂ + (sqrt(3)*x/2 - y/2) 𝐞₀₁₃ + (x/2 + sqrt(3)*y/2) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p13 = (-z) 𝐞₀₁₂ + (-sqrt(3)*x/2 - y/2) 𝐞₀₁₃ + (x/2 - sqrt(3)*y/2) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p14 = (sqrt(2)*x/3 + sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (sqrt(3)*x/6 - y/2 + sqrt(6)*z/3) 𝐞₀₁₃ + (-5*x/6 + sqrt(3)*y/6 + sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p21 = (-2*sqrt(2)*x/3 + z/3) 𝐞₀₁₂ + (sqrt(3)*x/6 + y/2 + sqrt(6)*z/3) 𝐞₀₁₃ + (x/6 - sqrt(3)*y/2 + sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

...

'p24 = (sqrt(2)*x/3 + sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (sqrt(3)*x/3 - sqrt(6)*z/3) 𝐞₀₁₃ + (2*x/3 - sqrt(3)*y/3 + sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p31 = (sqrt(2)*x/3 - sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (sqrt(3)*x/2 + y/2) 𝐞₀₁₃ + (x/6 - sqrt(3)*y/6 - 2*sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p32 = (sqrt(2)*x/3 - sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (-sqrt(3)*x/3 + sqrt(6)*z/3) 𝐞₀₁₃ + (2*x/3 + sqrt(3)*y/3 + sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p33 = (sqrt(2)*x/3 - sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (-sqrt(3)*x/6 - y/2 - sqrt(6)*z/3) 𝐞₀₁₃ + (-5*x/6 - sqrt(3)*y/6 + sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

'p34 = (sqrt(2)*x/3 + sqrt(6)*y/3 + z/3) 𝐞₀₁₂ + (-sqrt(3)*x/2 + y/2) 𝐞₀₁₃ + (x/6 + sqrt(3)*y/6 - 2*sqrt(2)*z/3) 𝐞₀₂₃ + (1) 𝐞₁₂₃'

In [8]:


# At this point we can formulate a very simple equilibrium condition that is needed for the tensegrity equilibirum 
# In order to achieve a stable vectorequilibrium at each point the compression vector in the direction of the center of the plate and the three cables must be in equilibrium. 
# If we substitute the two cable vectors by their resultant vector, the equilibrium is between tree vectors, i.e. all vectors and all directions are coplanar. 
# This allows us to use a coplanarity constraint wit four points as a basic requirement to achieve equlibrium:
# p11 - our main point of interest. 
# r1m - the center of an adjacent cable ring at p11. This point represents the direction of the resultant vector of the two ring-vectors at p11.
# p23 - the other endpoint of the edge-cable at p11
# q1m the center of the adjacent triangular polygon or plate.  

# By creating a regressive product of the four points in Geometric Algebra we get the signed volume of the tetrahedron between four points
# If the points are coplanar the volume of this signed volume of the tetrahedron is zero. 
# So equating the regressive product or our four points to zero is an algebraic expression for the coplanarity constraint on the four points.

eq1 = r1m & q1m & p23 & p11
eq1 = eq1.values()[0]
 # some formal simplification to get nicer numbers
eq1 = eq1 *3/4
# This leads to an cubic equation in x, y, z:
display(eq1)

-sqrt(6)*x**3 - 3*sqrt(2)*x**2*y + sqrt(3)*x**2*z + 12*x*y*z + sqrt(6)*x*z**2 - 3*sqrt(3)*y**2*z + 3*sqrt(2)*y*z**2

In [9]:
# In order to get cartesian distances of points in PGA (3,0,1), we need need to make sure they are normalized.
r1m = r1m.normalized()
q1m = q1m.normalized()

In [10]:
# We knock out one of three variables or degrees of freedom of eq1 by fixing the radius of the plate to 1
# This means that our triangular plates are fixed to have a circumradius of 1. 
# If want bigger or smaller plates later we can linearily scale our result. 
# We need a small subroutine to calculate the distance between two points in PGA (3,0,1)
# Since we dont want to deal with square roots more than necessary in our further algebraic derivations we use the squared distance of points.

def dist_sq(a, b): 
    return alg.rp(a, b).normsq().e

eq2 = dist_sq(p11,q1m)-1
eq2 = sym.nsimplify(eq2)

# We solve eq2 for y and substitute the solution in eq1 to get an equation with only two variables x and z.

y_solve = sym.solve(eq2,y)
eq1 = eq1.subs(y,y_solve[0])
display(eq1)

-sqrt(6)*x**3 + sqrt(3)*x**2*z - 3*sqrt(2)*x**2*sqrt(1 - x**2) + sqrt(6)*x*z**2 + 12*x*z*sqrt(1 - x**2) + 3*sqrt(2)*z**2*sqrt(1 - x**2) - 3*sqrt(3)*z*(1 - x**2)

In [11]:
# In order to specify a particular configuration we can knock out another variable by setting a rotation angle alpha of the plate
# This leads to some proportion of ring cables and edge cables representing a transformation from a tetrahedron into a cubeoctahedron

alpha = sym.symbols('alpha', real=True)
x_value = sym.cos(alpha)
y_value = sym.sin(alpha)
eq1_angle = eq1.subs(x,x_value)
eq1_angle = eq1_angle.subs(y,y_value)
eq1_anlge = eq1_angle.simplify()
display(eq1_angle)

3*sqrt(2)*z**2*sqrt(1 - cos(alpha)**2) + sqrt(6)*z**2*cos(alpha) + 12*z*sqrt(1 - cos(alpha)**2)*cos(alpha) - 3*sqrt(3)*z*(1 - cos(alpha)**2) + sqrt(3)*z*cos(alpha)**2 - 3*sqrt(2)*sqrt(1 - cos(alpha)**2)*cos(alpha)**2 - sqrt(6)*cos(alpha)**3

In [12]:
# Now we can get general algebraic expression for z with respect to a rotation-angle of the plate alpha.
# Since there is more than one solution we need to choose a pysically meaningful solution.
# With this equation we could can specifiy a rotation angle alpha to get a specific solution for x,y z. 

z_solve = sym.solve(eq1_angle,z)
z_solve[1]=z_solve[1].simplify()
z_solve[1]


(3*sqrt(-16*cos(alpha)**4 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) + 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 3) - 4*sqrt(3)*cos(alpha)**2 - 12*cos(alpha)*Abs(sin(alpha)) + 3*sqrt(3))/(2*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha))))

In [13]:
# We are down to only one variable the rotation angle alpha of the plate. 
# If we input a rotation angle between 0 and pi/3 degrees we can get specific values for x,y and z that represent a presstressable configuration of the tensegrity structure.
# Physically meaningful solutions for alpha are between 0 and pi/3 since the structure transforms from a tetrahedron into a cubeoctahedron in this range.

# alpha_r = np.pi/6
# We can make this visualization interactive by defining a function that takes the rotation angle alpha as an argument and updates the coordinates of the points and the graph function accordingly.
def visualize_transformation(alpha_slider, output_widget=None):
    
    # Interactive visualization of the tensegrity transformation as a function of rotation angle alpha. 
    if output_widget is not None:
        output_widget.clear_output(wait=True)
    
    # Calculate x, y, z based on alpha
    x_val = cos(alpha_slider)
    y_val = sin(alpha_slider)
    alpha_sym = sym.symbols('alpha')
    z_func_temp = sym.lambdify(alpha_sym, z_solve[1], modules='numpy')
    z_val = float(z_func_temp(alpha_slider))
    
    # Create the first point
    p11_c = point(x_val, y_val, z_val)
    
    # Generate all points using the symmetry function
    p12_c, p13_c, p21_c, p22_c, p23_c, p31_c, p32_c, p33_c, p14_c, p24_c, p34_c, r1m_c, q1m_c = points_tensegrity_tetrahedron(p11_c)
    
    # Naming the lines and plates of the structure for the current alpha
    # Ring cables
    s11_c = [p11_c.map(float), p21_c.map(float)]
    s12_c = [p21_c.map(float), p31_c.map(float)]
    s13_c = [p31_c.map(float), p11_c.map(float)]
    
    s21_c = [p12_c.map(float), p22_c.map(float)]
    s22_c = [p22_c.map(float), p32_c.map(float)]
    s23_c = [p32_c.map(float), p12_c.map(float)]
    
    s31_c = [p13_c.map(float), p23_c.map(float)]
    s32_c = [p23_c.map(float), p33_c.map(float)]
    s33_c = [p33_c.map(float), p13_c.map(float)]
    
    s41_c = [p14_c.map(float), p24_c.map(float)]
    s42_c = [p24_c.map(float), p34_c.map(float)]
    s43_c = [p34_c.map(float), p14_c.map(float)]
    
    # Edge cables
    ks1_c = [p11_c.map(float), p23_c.map(float)]
    ks2_c = [p21_c.map(float), p12_c.map(float)]
    ks3_c = [p31_c.map(float), p14_c.map(float)]
    ks4_c = [p22_c.map(float), p13_c.map(float)]
    ks5_c = [p33_c.map(float), p34_c.map(float)]
    ks6_c = [p24_c.map(float), p32_c.map(float)]

    # Polygon to show coplanarity constraint of the four points at p11. This is not a physical plate but a visual aid to show the coplanarity constraint that is needed for equilibrium at p11.
    coc_c = [[p11_c.map(float), r1m_c.map(float),q1m_c.map(float),p23_c.map(float)],[p11_c.map(float), r1m_c.map(float)], [r1m_c.map(float),q1m_c.map(float)],[q1m_c.map(float), p23_c.map(float)],[p23_c.map(float), p11_c.map(float)]]

    # Plates
    f1_c = [[p13_c.map(float), p12_c.map(float), p11_c.map(float)], [p13_c.map(float), p12_c.map(float)], [p12_c.map(float), p11_c.map(float)], [p11_c.map(float), p13_c.map(float)]]
    f2_c = [[p21_c.map(float), p14_c.map(float), p32_c.map(float)], [p21_c.map(float), p14_c.map(float)], [p14_c.map(float), p32_c.map(float)], [p32_c.map(float), p21_c.map(float)]]
    f3_c = [[p22_c.map(float), p24_c.map(float), p33_c.map(float)], [p22_c.map(float), p24_c.map(float)], [p24_c.map(float), p33_c.map(float)], [p33_c.map(float), p22_c.map(float)]]
    f4_c = [[p23_c.map(float), p34_c.map(float), p31_c.map(float)], [p23_c.map(float), p34_c.map(float)], [p34_c.map(float), p31_c.map(float)], [p31_c.map(float), p23_c.map(float)]]
    
    # Create the graph function with the title showing current alpha value
    def graph_func():
        return [
            "Tensegrity Tetrahedron Transformation (α = {:.1f}°)".format(alpha_slider * 180 / pi),
          
            '<G opacity="0.4">',
            0xaa999999, f1_c[0],
            '</G>',  

            '<G opacity="0.1">',
            0x008800, coc_c[0],
            '</G>',

            0xaa999999, f2_c[0],  
            0xaa999999, f3_c[0],
            0xaa999999, f4_c[0],
            
            0x00aa00, coc_c[1],
            0x00aa00, coc_c[2],
            0x00aa00, coc_c[3],
            0x00aa00, coc_c[4],

            0x00aa00, p11_c.map(float), "p11",
            0x00aa00, p23_c.map(float), "p23",
            0x00aa00, r1m_c.map(float), "center_r1",
            0x00aa00, q1m_c.map(float), "center_p1",
            0x555555, p12_c.map(float), "p12",
            0x555555, p13_c.map(float), "p13",
            0x555555, p21_c.map(float), "p21",
            0x555555, p22_c.map(float), "p22",
            0x00aa00, p23_c.map(float), "p23",
            0x555555, p31_c.map(float), "p31",
            0x555555, p32_c.map(float), "p32",
            0x555555, p33_c.map(float), "p33",
            0x555555, p14_c.map(float), "p14",
            0x555555, p24_c.map(float), "p24",
            0x555555, p34_c.map(float), "p34",
            
            0x555555, s11_c, "r11",
            0x555555, s12_c, "r12",
            0x555555, s13_c, "r13",
            0x555555, s21_c, "r21",
            0x555555, s22_c, "r22",
            0x555555, s23_c, "r23",
            0x555555, s31_c, "r31",
            0x555555, s32_c, "r32",
            0x555555, s33_c, "r33",
            0x555555, s41_c, "r41",
            0x555555, s42_c, "r42",
            0x555555, s43_c, "r43",
            
            0x555555, ks1_c, "e1",
            0x555555, ks2_c, "e2",
            0x555555, ks3_c, "e3",
            0x555555, ks4_c, "e4",
            0x555555, ks5_c, "e5",
            0x555555, ks6_c, "e6",
            
            0x555555, f1_c[1], f1_c[2], f1_c[3],
            0x555555, f2_c[1], f2_c[2], f2_c[3],
            0x555555, f3_c[1], f3_c[2], f3_c[3],
            0x555555, f4_c[1], f4_c[2], f4_c[3],

            0xff0000, p0.map(float), "center",  

        ]
    # Some setup for camera 
    # Rotors use half-angles:
    # Angles in radians: 30° = pi/6, 15° = pi/12
    # Rotors use half-angles:
    hx = 0  # float((sym.pi /18 )/ 2)   # Half of 45°
    hy = 0  # float((sym.pi /18) / 2)   # Half of 45°
    hz = 0 # float((sym.pi / 2)/2)
    
    # Some setup for camera 
    # Rotors use half-angles:

    Rx = cos(hx) - sin(hx) * e23  # Rotate around X (YZ plane)
    Ry = cos(hy) - sin(hy) * e13  # Rotate around Y (ZX plane)
    Rz = cos(hz) - sin(hz) * e12  # Rotate around Z (XY plane)
    
    # Multiply rotors to get the final camera perspective
    camera_motor = Rz * Ry * Rx
   
    return alg.graph(
        graph_func,
        camera=camera_motor,
        lineWidth=1,
        grid=False,
        labels=False,
        pointRadius=0.5,
        gl=0,
        fontSize=0.6,
        scale=1.1,
        height='1000px'
    )

# Create the slider for alpha (rotation angle)
alpha_slider = FloatSlider(
    value=pi/6,
    min=0,
    max=pi/3,
    step=0.01,
    description='α (rad):',
    continuous_update=False
)

# Create an output widget to display the graph
output_widget = ipywidgets.Output()

# Function to update the graph when slider changes
def on_slider_change(change):
    with output_widget:
        output_widget.clear_output(wait=True)
        # Capture the returned graph and explicitly display it
        graph = visualize_transformation(change['new'], output_widget=None)
        display(graph)

# Observe slider changes
alpha_slider.observe(on_slider_change, names='value')

# Display slider and output together
container = ipywidgets.VBox([alpha_slider, output_widget])
display(container)

# Initial plot - render the initial graph
with output_widget:
    # Capture the returned graph and explicitly display it
    graph = visualize_transformation(pi/6, output_widget=None)
    display(graph)

In [14]:
# Alternatively we may look at the proportions of ring-cables and edge-cables.
# Again we use squared lenghts for the proportions of ring-cables and edge-cables to avoid dealing with to many square roots.

edge_length = dist_sq(p11,p23)
ring_length = dist_sq(p11,p31)
proportion_sq = edge_length/ring_length
proportion_sq = proportion_sq.simplify()
proportion_sq



4*(x**2 - 2*sqrt(2)*x*z + 3*y**2 + 2*z**2)/(7*x**2 - 4*sqrt(3)*x*y - 2*sqrt(2)*x*z + 3*y**2 - 2*sqrt(6)*y*z + 8*z**2)

In [15]:
# we can substitute y with  sqrt(1-x^2) 
proportion_sq = proportion_sq.subs(y, sym.sqrt(1 - x**2))
proportion_sq = proportion_sq.simplify()
proportion_sq

4*(-2*x**2 - 2*sqrt(2)*x*z + 2*z**2 + 3)/(4*x**2 - 2*sqrt(2)*x*z - 4*sqrt(3)*x*sqrt(1 - x**2) + 8*z**2 - 2*sqrt(6)*z*sqrt(1 - x**2) + 3)

In [16]:
# again we can substitute x and y with the rotation angle alpha to get a formula for the cable proportion as a function of the rotation angle of the plate.
proportion_sq = proportion_sq.subs(x, sym.cos(alpha))
proportion_sq = proportion_sq.simplify()
proportion_sq



4*(2*z**2 - 2*sqrt(2)*z*cos(alpha) - cos(2*alpha) + 2)/(8*z**2 - 2*sqrt(2)*z*cos(alpha) - 2*sqrt(6)*z*Abs(sin(alpha)) + 4*cos(alpha)**2 - 4*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 3)

In [17]:
# since we have a formula for z we can substitute z. This will give us a formula for cable proportion as a function of the rotation angle alpha with a fixed circumradius of the plate.
proportion_sq = proportion_sq.subs(z, z_solve[1])
proportion_sq = proportion_sq.simplify()
proportion_sq



(2*(2 - cos(2*alpha))*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha)))**2 - 2*sqrt(2)*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha)))*(3*sqrt(-16*cos(alpha)**4 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) + 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 3) - 4*sqrt(3)*cos(alpha)**2 - 12*cos(alpha)*Abs(sin(alpha)) + 3*sqrt(3))*cos(alpha) + (3*sqrt(-16*cos(alpha)**4 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) + 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 3) - 4*sqrt(3)*cos(alpha)**2 - 12*cos(alpha)*Abs(sin(alpha)) + 3*sqrt(3))**2)/(9*(-32*(1 - cos(alpha)**2)**2 - 2*sqrt(3)*sqrt(-16*(1 - cos(alpha)**2)**2 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) - 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 19)*cos(alpha)**2 - 10*sqrt(-16*(1 - cos(alpha)**2)**2 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) - 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 19)*cos(alpha)*Abs(sin(alpha)) + sqrt(3)*sqrt(-16*(1 - cos(alpha)**2)**2 + 16*sqrt(3)*cos(alpha)**3*Abs(si

In [ ]:
# Since this is to hard to invert algebraically we can use numerical methods to solve for alpha given a specific cable proportion.
# We turn the symbolic expression into a fast numerical function
get_prop_num = sym.lambdify(alpha, proportion_sq, modules='numpy')
def solve_alpha_numeric(target_prop_sq):
    # We want to find alpha where: actual_proportion(alpha) - target_proportion = 0
    def objective_func(a):
        return get_prop_num(a) - target_prop_sq 
    try:
        # root_scalar uses the 'brentq' method to instantly hone in on the exact angle.
        # We restrict the search bracket between 0 and pi/3 (0 to 60 degrees).
         # Adding a tiny offset so we never evaluate exactly at 0 or pi/3
        epsilon = 1e-6
        lower_bound = 0.0 + epsilon
        upper_bound = np.pi/3 - epsilon
        res = root_scalar(objective_func, bracket=[lower_bound, upper_bound], method='brentq')
        return res.root
    except ValueError:
        # If the proportion is physically impossible (out of bounds), return 0
        return 0.0



Für das Verhältnis von Ringseilen zu Eckseilen = 1, ist der Rotationswinkel alpha = 0.48125622 radians (27.573950°)


In [68]:
# Now we can input a specific cable proportion to get the corresponding rotation angle alpha and the coordinates of the points for this specific configuration.
proportion_ring_edge = 1
c_prop_sq = proportion_ring_edge**2
alpha_num = solve_alpha_numeric(c_prop_sq)
x = cos(alpha_num)
y= sin(alpha_num)
z = z_solve[1].subs(alpha, alpha_num)
p11 = point(x,y,z)
p12, p13, p21, p22, p23, p31, p32, p33, p14, p24, p34, r1m, q1m = points_tensegrity_tetrahedron(p11)
plate_length = dist_sq(p11,p13)
edge_length = dist_sq(p11,p23)
ring_length = dist_sq(p11,p31)
# print (plate_length.evalf(), edge_length.evalf(), ring_length.evalf())




In [67]:

prop_ring_plate = (sqrt(ring_length.evalf())/sqrt(plate_length.evalf()))
prop_edge_plate = (sqrt(edge_length.evalf())/sqrt(plate_length.evalf()))

print(f"For a ratio of ring-cables to edge-cables = {proportion_ring_edge}, rotation angle alpha = {alpha_num:.8f} radians ({np.degrees(alpha_num):.6f}°)")
print(f"The ratio of ring-cables to plate-edges is = {prop_ring_plate}, and the ratio of edge-cables to plate-edges is = {prop_edge_plate}")


For a ratio of ring-cables to edge-cables = 1.5, rotation angle alpha = 0.59962648 radians (34.356067°)
The ratio of ring-cables to plate-edges is = 0.445145467251504, and the ratio of edge-cables to plate-edges is = 0.6677182008774477
